# GPM accuracy-time evaluation

**Before running:**

1. In Google Drive, create a folder called `ada_gpm` directly inside My Drive.
2. Upload `ada_colab_bundle.zip` and `epoch_380.pt` into that folder.
3. In Colab, go to **Runtime > Change runtime type**, choose **T4 GPU**, and save.
4. Run the cells below in order, one at a time. Allow Drive access when asked.

If a cell fails, copy its full output and paste it to Claude.


## 1. Connect Drive and unpack the project


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, zipfile
DRIVE = '/content/drive/MyDrive/ada_gpm'
CHECKPOINT = f'{DRIVE}/epoch_380.pt'
BUNDLE = f'{DRIVE}/ada_colab_bundle.zip'
PROJECT = '/content/ada'

for path in (BUNDLE, CHECKPOINT):
    if not os.path.exists(path):
        raise FileNotFoundError(f'Missing {path}. Upload it to the ada_gpm folder in Drive.')

zipfile.ZipFile(BUNDLE).extractall(PROJECT)
EVAL = f'{PROJECT}/accuracy_efficiency_eval'
PREDICTIONS = f'{PROJECT}/sampling_benefit_diagnostic/results/predictions.npz'
print('Project ready.')


## 2. Check the GPU


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv


## 3. Install GPM packages

This takes a few minutes. It must end with `SETUP OK`.


In [ ]:
!python {EVAL}/colab_setup.py


## 4. Run the project tests

This must end with `OK`.


In [ ]:
%cd {EVAL}
!python -m unittest -q test_evaluate


## 5. Measure real time

This downloads Computers, then times K = 4 to 128. Results save to Drive as `timing.json`.


In [ ]:
%cd {EVAL}
!python measure_timing.py --gpm-dir {PROJECT}/GPM/GPM --checkpoint {CHECKPOINT} --expected-predictions {PREDICTIONS} --output {DRIVE}/timing.json


## 6. Check the timing run

This checks that the run matches your saved results before they are used.


In [ ]:
import json
timing = json.load(open(f'{DRIVE}/timing.json'))
print('Device:', timing['device'], '| torch:', timing['torch'])
for r in timing['per_k']:
    print(f"K={r['k']:>3}  accuracy={r['accuracy']:.4f}  ms/node={r['total_s_per_node']*1000:.4f}  peak GPU MB={r['peak_gpu_mb']}")
gap = timing['saved_prediction_check']['largest_gap']
print(f'Largest accuracy gap from saved results: {gap*100:.2f} points')
print('CHECK PASSED' if gap < 0.01 else 'CHECK FAILED: do not use these results. Paste this output to Claude.')


## 7. Evaluate with real time

This combines real time with the saved predictions.


In [ ]:
%cd {EVAL}
!python evaluate.py --predictions {PREDICTIONS} --timing {DRIVE}/timing.json --output {DRIVE}/results_timed
!python plot_results.py --results {DRIVE}/results_timed
from IPython.display import Image, display
display(Image(f'{DRIVE}/results_timed/accuracy_vs_cost.png'))


## 8. Finish

Copy all the output from cells 6 and 7 and paste it to Claude. Your results are saved in `ada_gpm` in Drive, so nothing is lost when Colab closes.


## Later: add the final results

Run this only after Arvind shares his files. Upload `final_predictions.npz` and `adaptive_decisions.npz` to `ada_gpm`, and rerun cell 1 first.


In [ ]:
FINAL = f'{DRIVE}/final_predictions.npz'
DECISIONS = f'{DRIVE}/adaptive_decisions.npz'
%cd {EVAL}
!python evaluate.py --predictions {FINAL} --timing {DRIVE}/timing.json --adaptive-decisions {DECISIONS} --output {DRIVE}/results_final
!python plot_results.py --results {DRIVE}/results_final
display(Image(f'{DRIVE}/results_final/accuracy_vs_cost.png'))
